# AIC2026 — ma hoa nhom L26 (ViT-gopt-16-SigLIP2-384)

L26 la **45% ca kho**: 498 video, 79.590 anh. Day la nhom cuoi cung, va la nhom
duy nhat chua ai soi duoc bang mat — de so tuyen dot 2 co 5 cau Q&A roi vao no.

## Settings BAT BUOC

| | |
| --- | --- |
| Accelerator | **GPU T4 x2** — DUNG chon P100 (sm_60, torch chi build sm_70+) |
| Internet | **ON** |
| Input | `aic2026-index` **+ `quockhanhai/aic2026-keyframes-l26`** |

⚠️ Notebook nay chi can HAI dataset: `aic2026-index` va dataset L26.
Khong can 9 dataset anh kia — L26 nam rieng mot cho.

## 1. Ma nguon

In [ ]:
!rm -rf /tmp/repo
!git clone -q -b giai-doan-0 https://github.com/QuocKhanhDev-it/AIC_2026_FirstDance.git /tmp/repo
%cd /tmp/repo
!pip -q install open_clip_torch pandas pyarrow

## 2. Chot GPU — chay TRUOC khi tai 7,49 GB trong so

In [ ]:
import torch
assert torch.cuda.is_available(), "khong co GPU — kiem Settings > Accelerator"
ten = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
ho_tro = torch.cuda.get_arch_list()
print(f"GPU: {ten}  sm_{cc[0]}{cc[1]}")
assert f"sm_{cc[0]}{cc[1]}" in ho_tro, (
    f"{ten} (sm_{cc[0]}{cc[1]}) khong nam trong build torch {ho_tro}. "
    "Doi Accelerator sang T4 x2. P100 la sm_60, khong chay duoc.")
print("OK")

## 3. Chep `index/` va va duong dan

TIM file chu KHONG doan duong dan — Kaggle mount o
`/kaggle/input/datasets/<user>/<slug>/`, khong phai `/kaggle/input/<slug>/`.

In [ ]:
import glob, shutil, os, pathlib
print("co trong /kaggle/input:", os.listdir('/kaggle/input'))
pathlib.Path('index').mkdir(exist_ok=True)
for ten in ('master.parquet', 'clip.npy', 'trung_lap.parquet'):
    hit = glob.glob(f'/kaggle/input/**/{ten}', recursive=True)
    if hit:
        shutil.copy(hit[0], f'index/{ten}')
        print(f"  {ten:20} <- {hit[0]}")
    else:
        print(f"  {ten:20} KHONG THAY")
assert os.path.exists('index/master.parquet'), "thieu master.parquet"

In [ ]:
!python scripts/12_va_duong_dan.py --roots /kaggle/input --ghi

### Ham `chay()` — `!lenh` that bai KHONG lam dung notebook

In [ ]:
import subprocess

def chay(lenh):
    print("$", lenh, flush=True)
    p = subprocess.run(lenh, shell=True, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"ma thoat {p.returncode}: {lenh}")
    return p.stdout

### Chot chan — L26 phai co DU 79.590 anh

Khong phai kiem lay le. Neu dataset L26 la mot ban TRICH XUAT KHAC cua cung
video thi duong dan van khop (`va()` khop theo ten file = so thu tu keyframe),
so dong van du, **khong co gi bao**, va moi `frame_idx` nop cho L26 deu sai.

Da doi chieu so keyframe cua 57 video dau voi bang cai: khop 57/57. Cell nay
kiem lai tong so tren chinh may dang chay.

In [ ]:
import pandas as pd
m = pd.read_parquet('index/master.parquet')
co = m[m.kf_path.notna()]
n26 = int((co.video_id.str[:3] == 'L26').sum())
v26 = int(co[co.video_id.str[:3] == 'L26'].video_id.nunique())
print(f"L26 co anh: {n26:,} / 79.590   ({v26} / 498 video)")
assert n26 == 79_590, (
    f"L26 chi thay {n26:,} anh — chua Add Input dataset L26, "
    "hoac dataset chua giai nen xong.")
print("\nDU 79.590 anh — dung ban trich xuat cua BTC.")

## 4. Phan cua toi — DOI DUNG MOT SO

L26 chia RIENG voi 9 nhom kia. Ly do: L26 len Kaggle **sau** khi ca nhom da
chia xong phan con lai — chia lai toan kho tu dau se xao tron moi phan da giao,
va ai encode xong lai phai lam lai tu dau.

| phan | video | anh | phut @9,4 |
| ---: | ---: | ---: | ---: |
| 1 | 83 | 13.287 | 24 |
| 2-5 | 83 | ~13.260 | 24 |
| 6 | 83 | 13.259 | 24 |

Lech 28 anh giua nguoi nhieu nhat va it nhat.

Lam MOT MINH thi de `SO_NGUOI = 1` (ca 79.590 anh, ~2,4 gio — van vua mot phien).

In [ ]:
PHAN_CUA_TOI = 1        # <-- DOI SO NAY, tu 1 den 6
SO_NGUOI = 6            # de 1 neu muon lam het trong mot phien

import importlib.util, pandas as pd
_s = importlib.util.spec_from_file_location(
    "chia_viec", "scripts/47_chia_viec_encode.py")
_cv = importlib.util.module_from_spec(_s); _s.loader.exec_module(_cv)

m = pd.read_parquet('index/master.parquet')
m26 = m[m.video_id.str[:3] == 'L26']
phan, tai = _cv.chia(m26, SO_NGUOI)
v = phan[PHAN_CUA_TOI - 1]

TEN_DS = f'L26_phan_{PHAN_CUA_TOI}.txt'
RA = f'/kaggle/working/clip_gopt_L26_phan{PHAN_CUA_TOI}.npy'
open(TEN_DS, 'w').write('\n'.join(v) + '\n')
print(f"L26 phan {PHAN_CUA_TOI}/{SO_NGUOI}: {len(v)} video, "
      f"{tai[PHAN_CUA_TOI-1]:,} anh")
print(f"  cac phan khac: {tai}")
print(f"  ra: {RA}")

## 5. Encode

`--workers 4` vi Kaggle cap 4 vCPU. Tran VRAM thi ha `--batch` xuong 16.
Toc do da do: **9,4-11,4 anh/giay** tren T4.

In [ ]:
chay(f"python scripts/08_encode.py --model ViT-gopt-16-SigLIP2-384 --pretrained webli "
     f"--chi-video {TEN_DS} --workers 4 --batch 32 --out {RA}")

## 6. Kiem lech hang — quan trong hon binh thuong o day

Voi 9 nhom kia, anh nam tren may da dung index nen chac chan la anh cua BTC.
L26 thi den tu mot dataset nguoi khac day len, nen phep kiem nay lam LUON HAI
viec: bat lech hang, VA xac nhan anh dung la ban trich xuat cua BTC.

Ma tran chi co L26 nen ca 200 cap mau deu tu L26 — `trung_lap.parquet` co
1.624 cap L26 o cos >= 0,99, thua suc.

Anh khac thi cac cap trung lap het trung va phep kiem TRUOT ngay.

In [ ]:
chay(f"python scripts/08_encode.py --kiem-lech-hang {RA}")

## 7. Soat truoc khi tai ve

Tai `clip_gopt_L26_phan<N>.npy` **va** file `.json` cung ten.

> ⚠️ **KHONG tai `master.parquet` ve.** No da bi buoc 3 va thanh duong dan
> `/kaggle/input/...`; de len may local la moi thu doc anh chet hang loat.

> 💡 **Nen NEN `.npy` + `.json` vao mot file zip truoc khi gui.** Da can that
> 30/08: file `.json` nho di qua Drive bi mat het dau ngoac kep, va sidecar
> hong nghia la mat luon chot cung-model luc ghep.

In [ ]:
import numpy as np, glob, os
for f in sorted(glob.glob('/kaggle/working/*.np[yz]')):
    ten = os.path.basename(f)
    try:
        a = np.load(f, mmap_mode='r')
        if a.ndim != 2:
            print(f"{ten:34} {str(a.shape):18} (khong phai ma tran)")
            continue
        print(f"{ten:34} {str(a.shape):18} {a.dtype}  "
              f"co vector: {int((np.abs(a).sum(1) > 0).sum()):,}")
    except Exception as e:
        print(f"{ten:34} doc khong duoc: {type(e).__name__}")
print()
!cd /kaggle/working && zip -q -j nop_ve.zip clip_gopt_L26_phan*.npy clip_gopt_L26_phan*.json && ls -la nop_ve.zip